# Baseline 2 — Alamri et al. (2019): Two-feature ensemble
**Reference:** Alamri et al., ITS 2019 (LNCS 11528). 

DOI: 10.1007/978-3-030-22244-4_20---
### What this reproduces
Alamri's claim: *just two coarse week-1 engagement signals are enough forensemble trees to predict dropout*. We pick the two MOOCCubeX analogues —**video count** and **comment count** — restricted to weeks `1..W`.
### V2 setup notes (this re-run)
- Reads from **`dbo.tbl_graphgrued_weekly_active_base_balanced`** (V4 cohort + balanced label).
- Uses **TRUE weekly columns** `*_week1..*_week4` for windowed views (no linear scaling).
- Targets: `y_5class` (A=0..E=4) and `y_binary` (is_completed) — both already in V4.
- Stratified 8:1:1 split (seed=42), median-impute + standardize fit on train only.
### Model recipe- Models: RandomForest, AdaBoost, GradientBoosting, XGBoost (if installed).
- Inputs: only `video_count_week*` + `comment_count_week*` up to week W (sum).

In [3]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)

MODEL_BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
WINDOWS = [1, 2, 4]
LABEL_ORDER_5 = ["A", "B", "C", "D", "E"]

print("Imports OK — reading from", MODEL_BASE_TABLE)


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 22, Finished, Available, Finished, False)

Imports OK — reading from dbo.tbl_graphgrued_weekly_active_base_ordered5


In [4]:
# ============================================================
# 1) LOAD MODEL BASE TABLE (V4 balanced)
# ============================================================
SAMPLE_FRACTION = 1.0
MAX_ROWS = 300_000

sdf = spark.table(MODEL_BASE_TABLE)
total_rows = sdf.count()
print(f"V4 cohort rows: {total_rows:,}")

if SAMPLE_FRACTION < 1.0:
    sdf = sdf.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

pdf = sdf.toPandas()
if len(pdf) > MAX_ROWS:
    g = pdf.groupby("risk_label_5_idx", group_keys=False)
    pdf = g.apply(lambda d: d.sample(
        n=max(1, int(len(d) * MAX_ROWS / len(pdf))),
        random_state=SEED)).reset_index(drop=True)

print(f"Working dataframe shape: {pdf.shape}")
print("\n5-class distribution after sampling:")
print(pdf["risk_label_5_idx"]
      .map({0:"A",1:"B",2:"C",3:"D",4:"E"})
      .value_counts(normalize=True).round(4)
      .reindex(LABEL_ORDER_5))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 23, Finished, Available, Finished, False)

V4 cohort rows: 100,409
Working dataframe shape: (100409, 71)

5-class distribution after sampling:
risk_label_5_idx
A    0.0222
B    0.0812
C    0.0769
D    0.6411
E    0.1785
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 2) LABELS — already in V4; just alias
# ============================================================
pdf["y_5class"] = pdf["risk_label_5_idx"].astype(int)
pdf["y_binary"] = pdf["is_completed"].fillna(0).astype(int)

print("y_5class distribution:")
counts5 = pdf["y_5class"].value_counts().sort_index()
print(counts5)
min_share = counts5.min() / counts5.sum()
if min_share < 0.005:
    raise RuntimeError(
        f"Class collapse: smallest class is {min_share:.2%} < 0.5%. "
        f"Re-run 01_v4_relabel_balanced.ipynb with adjusted thresholds.")

print("\ny_binary distribution:")
print(pdf["y_binary"].value_counts(normalize=True).round(4))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 24, Finished, Available, Finished, False)

y_5class distribution:
y_5class
0     2231
1     8156
2     7721
3    64375
4    17926
Name: count, dtype: int64

y_binary distribution:
y_binary
0    0.6692
1    0.3308
Name: proportion, dtype: float64


In [6]:
# ============================================================
# 3) FEATURE GROUPS — TRUE weekly columns + static
# ============================================================
def _by_suffix(suffix):
    return sorted([c for c in pdf.columns if c.endswith(suffix)])

WEEK_COLS = {
    1: _by_suffix("_week1"),
    2: _by_suffix("_week1") + _by_suffix("_week2"),
    3: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3"),
    4: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3") + _by_suffix("_week4"),
}

EXCLUDE = {
    "user_id", "course_id", "enroll_time",
    "is_completed", "completion_rate",
    "risk_label_5_name", "risk_label_5_idx",
    "label_D_flag", "label_E_flag",
    "y_binary", "y_5class",
    "total_4w_interactions", "post_w1_interactions", "perf_composite",
    "course_norm_score",
    "problems_passed", "course_raw_score_sum", "total_problems",
    "risk_label_5_surrogate",
}
ALL_WEEKLY = set().union(*WEEK_COLS.values())

STATIC_COLS = [
    c for c in pdf.columns
    if c not in EXCLUDE
    and c not in ALL_WEEKLY
    and pdf[c].dtype.kind in "biufc"
]

print(f"Static feature count: {len(STATIC_COLS)}")
print("Static cols:", STATIC_COLS[:20], "..." if len(STATIC_COLS) > 20 else "")
print()
for W, cols in WEEK_COLS.items():
    print(f"W={W}: {len(cols)} weekly columns up to and including week {W}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 25, Finished, Available, Finished, False)

Static feature count: 16
Static cols: ['year_of_birth', 'num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_count_total_4w', 'video_watch_time_total_4w', 'chapter_coverage_ratio_raw', 'chapter_coverage_ratio', 'avg_chapter_progress_4w', 'chapter_progress_ratio_raw', 'chapter_progress_ratio', 'activity_signal_4w', 'is_active_4w'] 

W=1: 9 weekly columns up to and including week 1
W=2: 18 weekly columns up to and including week 2
W=3: 27 weekly columns up to and including week 3
W=4: 36 weekly columns up to and including week 4


In [7]:
# ============================================================
# 4) BUILD WINDOW VIEWS — leakage-safe (only weeks 1..W visible)
# ============================================================
def make_window_view(W):
    cols = STATIC_COLS + WEEK_COLS[W]
    cols = [c for c in cols if c in pdf.columns]
    Xdf  = pdf[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce").astype(float)
    return Xdf, cols

window_views = {}
for W in WINDOWS:
    Xdf, cols = make_window_view(W)
    window_views[W] = {"X": Xdf, "cols": cols}
    nan_pct = Xdf.isna().mean().mean()
    print(f"W={W}: shape={Xdf.shape}, mean NaN ratio={nan_pct:.3%}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 26, Finished, Available, Finished, False)

W=1: shape=(100409, 25), mean NaN ratio=27.326%
W=2: shape=(100409, 34), mean NaN ratio=43.076%
W=4: shape=(100409, 52), mean NaN ratio=59.910%


In [8]:
# ============================================================
# 5) STRATIFIED 8:1:1 SPLIT (per label)
# ============================================================
def split_idx(y, test_size=0.1, val_size=0.1, seed=SEED):
    idx = np.arange(len(y))
    idx_tv, idx_te = train_test_split(
        idx, test_size=test_size, stratify=y, random_state=seed)
    val_rel = val_size / (1.0 - test_size)
    idx_tr, idx_va = train_test_split(
        idx_tv, test_size=val_rel, stratify=y[idx_tv], random_state=seed)
    return idx_tr, idx_va, idx_te

splits = {}
for label in ["y_binary", "y_5class"]:
    y = pdf[label].values
    splits[label] = split_idx(y)
    n_tr, n_va, n_te = (len(splits[label][0]), len(splits[label][1]), len(splits[label][2]))
    print(f"{label}: train={n_tr}, val={n_va}, test={n_te}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 27, Finished, Available, Finished, False)

y_binary: train=80327, val=10041, test=10041
y_5class: train=80327, val=10041, test=10041


In [9]:
# ============================================================
# 6) METRIC HELPERS — EarlyRisk-MOOC metric set
# ============================================================
def evaluate_classifier(y_true, y_pred, y_proba=None, classes=None, name=""):
    res = {"name": name}
    res["accuracy"]    = accuracy_score(y_true, y_pred)
    res["macro_f1"]    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    res["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if classes is None:
        classes = sorted(list(set(y_true) | set(y_pred)))
    per_class_f1 = f1_score(y_true, y_pred, average=None,
                            labels=classes, zero_division=0)
    for cls, f in zip(classes, per_class_f1):
        res[f"f1_{cls}"] = float(f)
    if y_proba is not None and len(classes) >= 2:
        try:
            if len(classes) == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr",
                    labels=classes, average="macro")
        except Exception as e:
            res["auc_error"] = str(e)
    return res


def flatten_result(res, name, classes_order, runtime, W, label_col):
    out = {"name": name, "label": label_col, "window": W,
           "runtime_sec": round(runtime, 2)}
    for k, v in res.items():
        if k.startswith("f1_") and k != "f1_score":
            try:
                idx = int(k.split("_")[1])
                out[f"f1_{classes_order[idx]}"] = v
            except (ValueError, IndexError):
                out[k] = v
        elif k in ("accuracy","macro_f1","weighted_f1","auc","auc_ovr"):
            out[k] = v
    return out


print("Metric helpers ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 28, Finished, Available, Finished, False)

Metric helpers ready


In [10]:
# ============================================================
# 7) IMPUTATION + SCALING (fit on TRAIN only)
# ============================================================
def fit_impute_scale(X_tr, X_te, X_va=None):
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr_i = imp.transform(X_tr)
    X_te_i = imp.transform(X_te)
    sc = StandardScaler().fit(X_tr_i)
    out = (sc.transform(X_tr_i), sc.transform(X_te_i))
    if X_va is not None:
        out = out + (sc.transform(imp.transform(X_va)),)
    return out

print("fit_impute_scale ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 29, Finished, Available, Finished, False)

fit_impute_scale ready


In [11]:
# ============================================================
# 8) ALAMRI MINIMAL FEATURES
# ============================================================
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)
XGB_AVAILABLE = False
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as e:
    print(f"xgboost not available ({e})")


def alamri_features_at_W(W):
    """Sum video_count and comment_count from week 1 .. W.
       Falls back to whatever weekly engagement signals exist."""
    candidates_video   = [f"video_count_week{t}" for t in range(1, W+1)]
    candidates_comment = [f"comment_count_week{t}" for t in range(1, W+1)]
    cv = [c for c in candidates_video   if c in pdf.columns]
    cc = [c for c in candidates_comment if c in pdf.columns]
    X = pd.DataFrame(index=pdf.index)
    if cv:
        X["video_count_sum"]   = pdf[cv].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    else:
        X["video_count_sum"]   = 0.0
    if cc:
        X["comment_count_sum"] = pdf[cc].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    else:
        X["comment_count_sum"] = 0.0
    return X.astype(float).values


def get_alamri_models():
    M = {
        "RandomForest":     RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
        "AdaBoost":         AdaBoostClassifier(n_estimators=200, random_state=SEED),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=SEED),
    }
    if XGB_AVAILABLE:
        M["XGBoost"] = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=SEED, n_jobs=-1, eval_metric="mlogloss",
            use_label_encoder=False, verbosity=0)
    return M


all_results = []
for label_col, classes in [("y_binary", [0,1]), ("y_5class", LABEL_ORDER_5)]:
    idx_tr, idx_va, idx_te = splits[label_col]
    y_all = pdf[label_col].values
    n_cls = len(classes)
    for W in WINDOWS:
        X = alamri_features_at_W(W)
        X_tr_s, X_te_s = fit_impute_scale(X[idx_tr], X[idx_te])
        for name, m in get_alamri_models().items():
            print(f"[{label_col}] W={W} {name} ...")
            t0 = time.time(); m.fit(X_tr_s, y_all[idx_tr]); rt = time.time()-t0
            y_pred = m.predict(X_te_s)
            y_proba = m.predict_proba(X_te_s) if hasattr(m, "predict_proba") else None
            res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                       classes=list(range(n_cls)), name=name)
            row = flatten_result(res, name, classes, rt, W, label_col)
            all_results.append(row)
            print(f"   acc={row.get('accuracy',0):.4f}  macroF1={row.get('macro_f1',0):.4f}  rt={row['runtime_sec']}s")

res_df = pd.DataFrame(all_results)
print("\n=== Alamri et al. 2019 — V4 benchmark ===")
print(res_df.round(4).to_string(index=False))
res_df.to_csv("/tmp/results_alamri_2019.csv", index=False)
print("\nSaved -> /tmp/results_alamri_2019.csv")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 30, Finished, Available, Finished, False)

[y_binary] W=1 RandomForest ...


   acc=0.6687  macroF1=0.4010  rt=22.36s
[y_binary] W=1 AdaBoost ...


   acc=0.6691  macroF1=0.4009  rt=12.84s
[y_binary] W=1 GradientBoosting ...


   acc=0.6691  macroF1=0.4011  rt=7.53s
[y_binary] W=1 XGBoost ...


...

   acc=0.6691  macroF1=0.2538  rt=25.61s
[y_5class] W=4 RandomForest ...


   acc=0.6678  macroF1=0.2594  rt=14.6s
[y_5class] W=4 AdaBoost ...


   acc=0.6680  macroF1=0.2612  rt=25.25s
[y_5class] W=4 GradientBoosting ...


   acc=0.6678  macroF1=0.2594  rt=48.92s
[y_5class] W=4 XGBoost ...


   acc=0.6679  macroF1=0.2595  rt=32.62s

=== Alamri et al. 2019 — V4 benchmark ===
            name    label  window  runtime_sec  accuracy  macro_f1  weighted_f1   f1_0   f1_1    auc   f1_A  f1_B  f1_C   f1_D   f1_E  auc_ovr
    RandomForest y_binary       1        22.36    0.6687    0.4010       0.5365 0.8014 0.0006 0.5511    NaN   NaN   NaN    NaN    NaN      NaN
        AdaBoost y_binary       1        12.84    0.6691    0.4009       0.5366 0.8017 0.0000 0.5518    NaN   NaN   NaN    NaN    NaN      NaN
GradientBoosting y_binary       1         7.53    0.6691    0.4011       0.5367 0.8017 0.0006 0.5509    NaN   NaN   NaN    NaN    NaN      NaN
         XGBoost y_binary       1         5.80    0.6690    0.4014       0.5369 0.8016 0.0012 0.5511    NaN   NaN   NaN    NaN    NaN      NaN
    RandomForest y_binary       2         8.01    0.6690    0.4008       0.5365 0.8016 0.0000 0.5557    NaN   NaN   NaN    NaN    NaN      NaN
        AdaBoost y_binary       2        14.04    0.6691  